# **Business Understanding**

Financial fraud detection is the process of identifying and preventing fraudulent activity in financial transactions, data, and other areas. The goal is to minimize financial losses, protect assets, and ensure regulatory compliance. Knowing that financial fraud is an increasingly prevalent issue, with organizations constantly seeking advanced solutions to detect and prevent suspicious activity. it is essential to have a system that can identify potential threats. This system should be able to track data and activity across all channels, including online, in-person, and over the phone.

Knowing tha such processes cannot be mannually managed by human efforts, many organisations and financial institutions are attempting to implement machine learning tools to track real-time transaction data to spot and prevent possible financial fraudulent activities. Considering such background context, I thought it would be a great practice for me to utilise the powerful tool of machine learning in solving such real-life issue.

# **Data Understanding**

This dataset was inspired by real-world transaction data but was generated synthetically to avoid privacy concerns. It includes key features that play a critical role in fraud detection, such as transaction amounts, device types, geographic locations, currency, card type, and a "fraud" label indicating whether a transaction is suspicious.

Dataset comprises of different categories of data columns:

1. **Comprehensive Transaction Categories**: Transactions span categories like retail (online and in-store), groceries, restaurants (fast food to premium), entertainment (streaming, gaming, events), healthcare, education, gas, and travel.

2. **Geographic and Demographic Variety**: The dataset includes diverse geographic data (countries, cities) and currency types, allowing for analysis on a global scale with varying risk profiles.

3. **Detailed Customer Profiles**: Each transaction is linked to a customer profile that includes characteristics like account age, preferred devices, typical spending range, and fraud-protection features.

# **About dataset**

Purpose of this project is followed by:
**Detecting whether such a transaction is fraudulent transaction activity or not** : using different categories of columns, I will be implementing different machine learning ensemble tools like XGBClassifier, Random Forest, Bagging, Voting, and Stacking

Following information is about each data column:
1. **Transaction ID** : Unique alphanumeric identifier for each transaction; useful for referencing individual transactions.
2. **Customer ID** : Unique ID assigned to each customer; helps track customer activity and behavior across transactions.
3. **Card Number** : A masked card number representing the credit or debit card used; ensures privacy while providing a unique identifier for card-level analysis.
4. **Timestamp** : Timestamp in UTC format indicating when the transaction occurred; facilitates time-based analysis, such as peak hours or fraud detection related to transaction timing.
5. **Merchant Category** : High-level category for the merchant, such as 'Retail' or 'Travel'; aids in identifying spending patterns by category.
6. **Merchant Type** : Specifies the subtype of merchant within each category, such as 'online' for Retail; useful for analyzing customer preferences for specific transaction types.
7. **Merchant** : Name of the merchant where the transaction took place; valuable for merchant-specific analysis, brand loyalty insights, and fraud detection.
8. **Amount** : Transaction amount in the local currency of the transaction's country; useful for monetary trend analysis and fraud detection.
9. **Currency** : Currency code (e.g., USD, EUR) used for the transaction; helps in currency-based aggregations or conversions for global analysis.
10. **Country** : Country where the transaction took place; essential for geographical analysis of spending and cross-border transaction risk assessment.
11. **City** : Name of the city where the transaction was made; provides additional granularity for regional analysis.
12. **City Size** : Classification of the city size (e.g., large, medium); helpful for understanding urban vs. rural transaction behavior.
13. **Card Type** : Type of card used in the transaction, such as 'Gold Credit' or 'Basic Debit'; used to assess patterns and fraud risks associated with different card types.
14. **Card Present** : Boolean indicating if the card was physically present during the transaction; important for differentiating between in-person and online transactions.
15. **Device** : Specific device or browser used for the transaction (e.g., Chrome, iOS App); aids in device-specific security and behavior analysis.
16. **Channel** : Specifies the transaction channel (web, mobile, pos); valuable for understanding channel preferences and detecting anomalies.
17. **Device Fingerprint** : Unique identifier for the device used, generated using hashing; useful in identifying fraudulent behavior across sessions.
18. **IP Address** : IP address used in the transaction, simulated for privacy; essential for tracking potential geo-fraud.
19. **Distance From Home** : Binary indicating whether the transaction occurred outside the customer's home country; can help in fraud detection by highlighting unusual travel patterns.
20. **High Risk Merchant** : Boolean that flags higher-risk merchant categories (e.g., Travel, Entertainment); useful for fraud risk modeling.
21. **Transaction Hour** : The hour (0–23) when the transaction occurred; allows for time-of-day analysis and detection of unusual transaction hours.
22. **Weekend Transaction** : Boolean indicating if the transaction occurred on a weekend; assists in examining behavioral patterns and fraud tendencies on weekends.
23. **Velocity Last Hour** : Dictionary of velocity metrics within the past hour, including: num_transactions: Number of customer transactions in the last hour. total_amount: Total amount spent in the last hour. unique_merchants: Count of unique merchants in the last hour. unique_countries: Count of unique countries in the last hour. max_single_amount: Maximum single transaction amount in the last hour.
24. **Is Fraud** : Binary label for fraud status (True/False); main target variable for fraud detection model training and validation.

# Import Packages

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, roc_curve, auc
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

In [ ]:
pd.set_option('display.max_columns', 500)

# Dataset Exploration

In [ ]:
transaction = pd.read_csv('/kaggle/input/transactions/synthetic_fraud_data.csv')
transaction.head()

In [ ]:
## Identify transaction data's information
transaction.info()

Data type of each column needs to be revised ex) timestamp, etc.

In [ ]:
## Explore statistical information about transaction data
transaction.describe()

Certainly both amount and transaction hour columns need to be investigated considering their mean value greater than the median value

In [ ]:
## Explore dimension of the data
print(transaction.shape)

seems like sampling is a must considering huge size of the dataset

In [ ]:
## sample the dataset
transaction_sample = transaction.sample(n = 250000, random_state = 21).reset_index(drop = True)

# Data Preprocessing

In [ ]:
transaction_sample.head()

In [ ]:
## Drop PII that might be unncessary in performing further regression task
transaction_sample.drop(['transaction_id', 'customer_id', 'card_number', 'ip_address', 'device_fingerprint'], axis = 1, inplace = True)

In [ ]:
## Convert Boolean values into int 
boolean_dtype = transaction_sample.select_dtypes('boolean').columns
for boolean_col in boolean_dtype:
    transaction_sample[boolean_col] = transaction_sample[boolean_col].astype('int')

In [ ]:
## Convert timestamp column into different data type
transaction_sample['timestamp'] = pd.to_datetime(transaction_sample['timestamp'], format='ISO8601')

In [ ]:
## Extract velocity last hour column values
import ast

# Parse all entries in 'velocity_last_hour' column once
parsed_data = []
for value in transaction_sample['velocity_last_hour']:
    try:
        # Parse each value as a dictionary
        parsed_data.append(ast.literal_eval(value) if isinstance(value, str) else value)
    except (ValueError, SyntaxError):
        # In case of parsing error, append a default dictionary with None values
        parsed_data.append({
            'num_transactions': None,
            'total_amount': None,
            'unique_merchants': None,
            'unique_countries': None,
            'max_single_amount': None
        })

# Convert parsed data to columns
transaction_sample['num_transactions'] = [int(item.get('num_transactions', 0)) for item in parsed_data]
transaction_sample['total_amount'] = [float(item.get('total_amount', 0)) for item in parsed_data]
transaction_sample['unique_merchants'] = [int(item.get('unique_merchants', 0)) for item in parsed_data]
transaction_sample['unique_countries'] = [int(item.get('unique_countries', 0)) for item in parsed_data]
transaction_sample['max_single_amount'] = [float(item.get('max_single_amount', 0)) for item in parsed_data]

# Drop velocity_last_hour column
transaction_sample.drop('velocity_last_hour', axis = 1, inplace = True)

In [ ]:
## Extract year, month, day, etc.
def modify_timestamp():
    transaction_sample['year'] = transaction_sample['timestamp'].dt.year
    transaction_sample['month'] = transaction_sample['timestamp'].dt.month
    transaction_sample['day'] = transaction_sample['timestamp'].dt.day
    transaction_sample['hour'] = transaction_sample['timestamp'].dt.hour
    transaction_sample['minute'] = transaction_sample['timestamp'].dt.minute
    transaction_sample['second'] = transaction_sample['timestamp'].dt.second
    transaction_sample['microsecond'] = transaction_sample['timestamp'].dt.microsecond
    transaction_sample.drop('timestamp', axis = 1, inplace = True)

modify_timestamp()

In [ ]:
transaction_sample.head() 

In [ ]:
transaction_sample.info()

In [ ]:
transaction_sample.isna().sum()

luckily no NA values !

In [ ]:
transaction_sample.duplicated().sum()

Same with duplicated values

Now it seems that the data has been well cleaned for further EDA

# EDA

In [ ]:
## Select only int or float valued columns
numerical_cols = transaction_sample.select_dtypes(include = ['int', 'float']).columns

## Remove those boolean valued columns converted into int & date related fields
numerical_cols = numerical_cols.drop([
    'card_present', 'distance_from_home', 'high_risk_merchant', 'weekend_transaction', 'is_fraud', 'year', 'month', 'day', 'hour', 'minute', 'second',
    'microsecond'
])

In [ ]:
transaction_sample['amount']

In [ ]:
# Adjust the number of rows and columns for subplots
ncol = 2
nrow = len(numerical_cols) // ncol + 1

fig, axes = plt.subplots(nrow, ncol, figsize=(20, 15))
axes = axes.flatten()  # Flatten axes for easier indexing

for i, col in enumerate(numerical_cols):
    sns.histplot(data=transaction_sample, x=col, ax=axes[i], kde = True)
    axes[i].set_title(f'Histogram for {col}')
    axes[i].set_ylabel('Frequency')

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Key things to note
* Certainly the amount column must be checked in terms of Boxplot considering its weird distribution
* number of transaction & total amount columns are rightly skewed and hence can be performed of log transformation for better modelling result
* unique merchants column is leftly skewed and can be performed of log transformation for similar purpose
* max single amount seems to have distributed normally but with some outliers so need to be checked in a boxplot

In [ ]:
# Adjust the number of rows and columns for subplots
ncol = 2
nrow = len(numerical_cols) // ncol + 1

fig, axes = plt.subplots(nrow, ncol, figsize=(20, 15))
axes = axes.flatten()  # Flatten axes for easier indexing

for i, col in enumerate(numerical_cols):
    sns.boxplot(data=transaction_sample, x=col, ax=axes[i])
    axes[i].set_title(f'Boxplot for {col}')
    
# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

what can be seen from this is that the numerical columns tend to possess large volume of outliers. So let's check the actual number of such outliers

In [ ]:
def check_outliers():
    for col in numerical_cols:
        q1 = transaction_sample[col].quantile(0.25)
        q3 = transaction_sample[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        
        outlier_count = len(transaction_sample[(transaction_sample[col] > upper) | (transaction_sample[col] < lower)])
        print(f'Outlier Count for {col} : {outlier_count}')

check_outliers()

Considering the entire dataset with 250000 data records, removing outliers would not affect the prediction by a lot. Nevertheless, taking the result of histogram also into apart, the data should rather go through **MinMaxScaler** to reduce the impact of outliers

In [ ]:
binary_cols = ['card_present', 'distance_from_home', 'high_risk_merchant', 'weekend_transaction', 'is_fraud']

# Adjust the number of rows and columns for subplots
ncol = 2
nrow = len(binary_cols) // ncol + 1

fig, axes = plt.subplots(nrow, ncol, figsize=(20, 30))
axes = axes.flatten()  # Flatten axes for easier indexing

for i, col in enumerate(binary_cols):
    value_counts = transaction_sample[col].value_counts(normalize=True)
    value_counts.plot.pie(ax=axes[i], autopct='%1.1f%%')  
    axes[i].set_title(f'{col} distribution')
    axes[i].set_ylabel('')  

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
## Get the name of entire categorical features
categorical_cols = list(transaction_sample.select_dtypes('object').columns)
categorical_cols.extend(['card_present', 'distance_from_home', 'high_risk_merchant', 'weekend_transaction'])
categorical_cols.remove('merchant')
categorical_cols

Knowing that all boolean columns are unbalanced, using > ***upsampling*** or > ***downsampling*** would help before modelling

In [ ]:
# Adjust the number of rows and columns for subplots
ncol = 4
nrow = len(categorical_cols) // ncol + 1

fig, axes = plt.subplots(nrow, ncol, figsize=(25, 18))
axes = axes.flatten()  # Flatten axes for easier indexing

for i, col in enumerate(categorical_cols):
    sns.histplot(data=transaction_sample, x=col, hue = 'is_fraud', multiple = 'dodge', shrink = 0.8, ax=axes[i])
    axes[i].set_title(f'{col} vs fraudulent transaction status')
    axes[i].tick_params(axis = 'x', rotation = 45)
    
# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Key things to note
1. no signficant difference among different merchant categories
2. online merchant type contained both the highest number of fraudulent and non fraudulent transaction -> indicates
online merchant goes through large volume of transaction
3. RUB, NGN, MXN, BRL are the currencies with large volume of fraudulent transactions while other
key currencies like euroes or usd go through way much less fraudulent transactions
4. Country follows similar trend like the currency -> Russia, Nigeria, Mexico, Brazil 
5. Unknown city contains the highest volume of fraudulent transactions while other cities in US contain 
dismissable amount / moreover, for large sized city contains almost no fraudulent transaction activities 
while medium sized city contains almost 50000
6. card type did not have a particular trend of fraudulent transaction activities as all card types showed similar distribution
7. for devices used in transaction, NFC Payment, magenetic stripe, and chip reader contained only fraudulent transactions
while other devices contained higher non fraudulent transactions
8. POS system contained only fraudulent transactions while others did not
9. comparing in person and online transaction, in person transaction only contained fraudulent transactions
10. fraudulent transaction activities were higher on home countries where its number exceeded non fraudulent activities


In [ ]:
plt.figure(figsize=(15, 10))
sns.scatterplot(data=transaction_sample, x='amount', y='max_single_amount', hue='is_fraud', alpha=0.7)
plt.title('Amount vs Maximum Single Transaction Amount in Last Hour on Different Fraudulent Transaction Status')
plt.xlabel('Amount')
plt.ylabel('Max Single Amount')
plt.show()

> ***No linear relationship can be spotted indicating that amount and max single amount columns are not correlated***

From the EDA of categorical variables, countries like Russia, Nigeria, Mexico, Brazil contained higher number of fraudulent transactions. So I will be exploring transactiosn specifically on such countries 

In [ ]:
## Specific exploration on particular countries with higher proportion of fraudulent transactions
country_specific = transaction_sample[transaction_sample['country'].isin(['Russia', 'Nigeria', 'Mexico', 'Brazil'])]
country_vis = country_specific.groupby(['country', 'is_fraud'])['amount'].agg('median').reset_index()
country_vis

In [ ]:
## Amount of Transaction compared among different countries with different fraud status
plt.figure(figsize=(10, 5))
sns.barplot(data = country_vis, x = 'country', y = 'amount', hue = 'is_fraud', palette = sns.color_palette("viridis"))
plt.title('Median Transaction Amount on Particular Countries')
plt.xlabel('Country')
plt.ylabel('Median Transaction Amount')
plt.show()

All those countries had higher transaction amount for fraudulent transactions compared to that of normal transactions. Among them, Nigeria had the highest transaction amount compared to other countries.  

In [ ]:
## Separate all transactions happened at Nigeria which its amount was abnormally high
nigeria_high_amount = transaction_sample[(transaction_sample['country'] == 'Nigeria') & (transaction_sample['amount'] > 294516)]
print(f'number of abnormal transactions at Nigeria : {nigeria_high_amount.shape[0]}')

In [ ]:
cols = ['distance_from_home', 'card_present']

## Visualisation
fig, axes = plt.subplots(1, 2, figsize = (15, 5))

for i, col in enumerate(cols):
    tmp_data = nigeria_high_amount[[col, 'is_fraud']].value_counts().reset_index()
    sns.barplot(tmp_data, x = col, y = 'count', hue = 'is_fraud', palette = sns.color_palette('rocket'), ax = axes[i])
    axes[i].set_title(f'Large Sized Transactions at Nigeria on {col}')

plt.tight_layout()
plt.show()

Clients who have transacted quite large amount which is greater than the median value of transaction amount at Nigeria are not originally from Nigeria, instead they just performed transaction at Nigeria and for those who performed transaction outside the home country at Nigeria have higher proportion of fraudulent transaction activities. Moreover offline transactions contained more proportion of normal transactions while online transaction only contained fraudulent transactions.

Let's check if other countries like Russia, Mexico and Brazil follow similar pattern.

In [ ]:
## Perform similar process for Brazil
brazil_high_amount = transaction_sample[(transaction_sample['country'] == 'Brazil') & (transaction_sample['amount'] > 3811)]
print(f'number of abnormal transactions at Brazil : {brazil_high_amount.shape[0]}')
cols = ['distance_from_home', 'card_present']

## Visualisation
fig, axes = plt.subplots(1, 2, figsize = (15, 5))

for i, col in enumerate(cols):
    tmp_data = brazil_high_amount[[col, 'is_fraud']].value_counts().reset_index()
    sns.barplot(tmp_data, x = col, y = 'count', hue = 'is_fraud', palette = sns.color_palette('mako'), ax = axes[i])
    axes[i].set_title(f'Large Sized Transactions at Brazil on {col}')

plt.tight_layout()
plt.show()

In [ ]:
## Perform similar process for Russia
russia_high_amount = transaction_sample[(transaction_sample['country'] == 'Russia') & (transaction_sample['amount'] > 56978)]
print(f'number of abnormal transactions at Russia : {russia_high_amount.shape[0]}')
cols = ['distance_from_home', 'card_present']

## Visualisation
fig, axes = plt.subplots(1, 2, figsize = (15, 5))

for i, col in enumerate(cols):
    tmp_data = russia_high_amount[[col, 'is_fraud']].value_counts().reset_index()
    sns.barplot(tmp_data, x = col, y = 'count', hue = 'is_fraud', palette = sns.color_palette('crest'), ax = axes[i])
    axes[i].set_title(f'Large Sized Transactions at Russia on {col}')

plt.tight_layout()
plt.show()

In [ ]:
## Perform similar process for Mexico
mexico_high_amount = transaction_sample[(transaction_sample['country'] == 'Mexico') & (transaction_sample['amount'] > 14976)]
print(f'number of abnormal transactions at Mexico : {mexico_high_amount.shape[0]}')
cols = ['distance_from_home', 'card_present']

## Visualisation
fig, axes = plt.subplots(1, 2, figsize = (15, 5))

for i, col in enumerate(cols):
    tmp_data = mexico_eda = mexico_high_amount[[col, 'is_fraud']].value_counts().reset_index()
    sns.barplot(tmp_data, x = col, y = 'count', hue = 'is_fraud', palette = sns.color_palette('cubehelix'), ax = axes[i])
    axes[i].set_title(f'Large Sized Transactions at Mexico on {col}')

plt.tight_layout()
plt.show()

Among those countries which had higher proportion of fraudulent transactions with its transaction amount greater than median transaction amount,  there has been more number of foreigners dealing with large amount of transactions that are fraudulent and for those transactions occured online were all fraudulent.

> **Fraudulent transactions found at Russia, Mexico, Nigeria, Brazil are done by foreigners at online platform at a large sized scale**

Next column we should explore considering previous result from analysis of categorical variables is > ***device***. This is because specifically devices like NFC Payment, magenetic stripe, and chip reader contained only fraudulent transactions while other devices contained higher non fraudulent transactions.

In [ ]:
## Exploring median amount of transaction for different devices and different transaction status
tr_amount_device = transaction_sample.groupby(['device', 'is_fraud'])[['amount']].agg('median').reset_index()
tr_amount_device

In [ ]:
## Visualisation
plt.figure(figsize = (10, 5))
sns.barplot(data = tr_amount_device, x = 'device', y = 'amount', hue = 'is_fraud')
plt.title('Median Transaction Amount for Different Devices & Transaction Status')
plt.ylabel('Median Transaction Amount')
plt.xticks(rotation = 45)
plt.show()

Regardless of device type, median amount of transaction is way much higher for fraudulent transactions than that of non fraudulent transactions. More specifically, Chip Reader, Magnetic Stripe and NFC Payment did not contain any normal transactions which should be explored more.

In [ ]:
## Exploring Chip Reader, Magnetic Stripe and NFC Payment
weird_device = transaction_sample[transaction_sample['device'].isin(['Chip Reader', 'Magnetic Stripe', 'NFC Payment'])]
weird_device.groupby('device')['is_fraud'].value_counts().reset_index()

> ***there are no normal transactions happened through Chip Reader, Magnetic Stripe and NFC Payment.***

In [ ]:
## Find out median transaction amount on different transaction hours
hour_med_amount = transaction_sample.groupby(['transaction_hour', 'is_fraud'])[['amount']].agg('median').reset_index()
hour_med_amount

In [ ]:
## Visualisation
plt.figure(figsize = (10, 5))
sns.lineplot(data = hour_med_amount, x = 'transaction_hour', y = 'amount', hue = 'is_fraud', linestyle = '-', marker = 'o',
            palette = ['orange', 'green'])
plt.axvline(x = 9, color = 'r', linestyle = '-.', linewidth = 2.5) ## Bank Opening Time
plt.axvline(x = 17, color = 'b', linestyle = '-.', linewidth = 2.5) ## Bank Closing Time
plt.legend(title = 'Fraudulent Transaction Status')
plt.title('Median Transaction Amount on Different Transaction Hour')
plt.xlabel('Transaction Hour')
plt.ylabel('Median Transaction Amount')
plt.show()

Median transaction amount has been always greater for fraudulent transactions than that of normal transactions regardless of transaction hour. During the bank operating hour (9AM - 5PM), the median transaction amount is the highest (at 3PM) but there is no significant trend on such bank operating hour affecting on transaction amount. 
> ***Transaction Amount is higher on fraudulent transactions than that of normal transactions***

In [ ]:
## Find out median transaction frequency on different transaction hours
hour_med_frequency = transaction_sample.groupby(['transaction_hour', 'is_fraud'])[['num_transactions']].agg('median').reset_index()
hour_med_frequency

In [ ]:
## Visualisation
plt.figure(figsize = (10, 5))
sns.lineplot(data = hour_med_frequency, x = 'transaction_hour', y = 'num_transactions', hue = 'is_fraud', linestyle = '-', marker = 'o',
            palette = ['orange', 'green'])
plt.axvline(x = 9, color = 'r', linestyle = '-.', linewidth = 2.5) ## Bank Opening Time
plt.axvline(x = 17, color = 'b', linestyle = '-.', linewidth = 2.5) ## Bank Closing Time
plt.legend(title = 'Fraudulent Transaction Status')
plt.title('Median Transaction Frequency on Different Transaction Hour')
plt.xlabel('Transaction Hour')
plt.ylabel('Median Transaction Frequency')
plt.show()

> ***No big significant trend on transaction frequency on and off bank operating time***

but key thing to notice is
* Transaction Frequency is extremely high for fraudulent transactions at the start of bank operating time
* Transaction Frequency is higher for normal transactions at the end of bank operating time

In [ ]:
## Compare general distribution of transaction frequencies and total amount spent last hour 
ex_cols = ['num_transactions', 'total_amount']

fig, axes = plt.subplots(1, 2, figsize = (15, 5))

for i, col in enumerate(ex_cols):
    sns.histplot(data = transaction_sample, x = col, hue = 'is_fraud', multiple = 'dodge', kde = False, palette = 'pastel', legend = True, 
                 element = 'bars', ax = axes[i])
    axes[i].set_title(f'Distribution of {col} on Transaction Status')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

> ***There seems to be no particular trend on distribution of transaction frequencies and total amount spent during last hours on different transaction status.***

Now I will perform steps on feature scaling, engineering, etc.

# Feature Engineering

In [ ]:
transaction_sample.head()

The current transaction amount is represented in its own local currency in which new feature should be generated using the standard USD

In [ ]:
transaction_sample['currency'].unique()

In [ ]:
def convert_currency_amt(transaction_sample):
    # Define the conversion rates for each currency
    conversion_rates = {
        'EUR': 1.06,
        'CAD': 0.72,
        'RUB': 0.01,
        'NGN': 0.0006,
        'SGD': 0.75,
        'MXN': 0.049,
        'BRL': 0.17,
        'AUD': 0.65,
        'JPY': 0.0065
    }
    
    # Add a column by mapping the currency to the corresponding conversion rate,
    # defaulting to 1.28 for Great Britain Pound if currency is not in the dictionary
    transaction_sample['USD_converted_amount'] = transaction_sample['amount'] * \
                                                 transaction_sample['currency'].map(conversion_rates).fillna(1.28)
    transaction_sample.drop(['amount', 'currency'], axis = 1, inplace = True)
    return transaction_sample

# Run the function
transaction_sample = convert_currency_amt(transaction_sample)

In [ ]:
transaction_sample['USD_converted_amount'].describe()

In [ ]:
plt.figure(figsize = (10, 5))
sns.histplot(data = transaction_sample, x = 'USD_converted_amount', kde = True)
plt.title('Distribution of Transaction Amount Converted to USD')
plt.show()

Now it seems that the distribution of transaction amount has come alright compared to how it was shown before during EDA

In [ ]:
## Add a new column indicating whether bank is operating or not
def is_bank_operating(transaction_sample):
    transaction_sample['is_bank_operating'] = np.where(
        transaction_sample['transaction_hour'] < 9, 0,
        np.where(transaction_sample['transaction_hour'] <= 17, 1, 0)
    )
    return transaction_sample

# Apply the function
transaction_sample = is_bank_operating(transaction_sample)

In [ ]:
## Check if there are no NA values
transaction_sample.isna().sum()

In [ ]:
## Check if any column has duplicating values
transaction_sample.duplicated().sum()

Considering 0 number of NA values along with 0 duplicating values, there is no need for iterative method to fill NAs

Now I will attempt to convert categorical variables into integers by dividing the categorical variables into ordinal and nomial categorical variables.

In [ ]:
transaction_sample['card_type'].unique()

In [ ]:
## Ordinal Categorical Variable
transaction_sample['card_type'] = transaction_sample['card_type'].map({
    'Premium Debit' : 4, 
    'Platinum Credit' : 3,
    'Gold Credit' : 2,
    'Basic Debit' : 1,
    'Basic Credit' : 0
})

In [ ]:
transaction_sample['merchant'].unique()

knowing that there are so many merchants, I decided to only limit top 20 merchants

In [ ]:
# Identify top merchants based on frequency or other criteria
top_merchants = transaction_sample['merchant'].value_counts().nlargest(20).index

# Replace merchants not in top_merchants with "Other"
transaction_sample['merchant'] = transaction_sample['merchant'].apply(lambda x: x if x in top_merchants else 'Other')

# One-hot encode
transaction_sample = pd.get_dummies(transaction_sample, columns=['merchant'], prefix='merchant', dtype = 'int')

In [ ]:
transaction_sample.head()

knowing that I will use Tree based models for this project, it would be better to use one hot encoding which the tree based models can deal with well. So I will continue to use one hot encoding on rest of the nominal categorical variables

In [ ]:
## Perform One-Hot Encoding on categorical variables
transaction_cleaned = pd.get_dummies(transaction_sample, dtype = 'int')

In [ ]:
transaction_cleaned.head()

In [ ]:
transaction_cleaned.isna().any(axis = 1).sum()

In [ ]:
transaction_cleaned.duplicated().sum()

In [ ]:
## Heatmap Creation to perform multivariate analysis
transaction_corr = transaction_cleaned.corr()

# Set up a threshold to limit uncorrelated values to be visible
threshold = 0.5

# Filter the possible correlations
filtered_corr = transaction_corr[abs(transaction_corr) >= 0.5]

# Visualisation
plt.figure(figsize = (20, 20))
sns.heatmap(filtered_corr, vmin = -1, vmax = 1, annot = True, fmt = '.2f', linewidth = 0.5, mask = filtered_corr.isnull(), 
           cmap = 'coolwarm', cbar_kws = {'label' : 'correlation'})
plt.show()

* **num_transactions weakly correlated with total amount and unique merchants, max single amount**

* **distance from home and card presence status is correlated with transaction status**s

* **distance from home is highly correlated with POS system**em


In [ ]:
## Select only necessary features with its threshold greater than 0.25
plt.figure(figsize = (15, 8))
transaction_corr['is_fraud'][:-1].abs().sort_values().plot(kind = 'bar', title = 'Most Important Features')
plt.show()

In [ ]:
## Select only 20 important features into list
selected_features = transaction_corr['is_fraud'][:-1].abs().sort_values().tail(22)
sel_feature_cols = selected_features.reset_index()['index']
sel_feature_cols

# Splitting Training & Testing Data

In [ ]:
## Define X & y variable
X = transaction_cleaned[sel_feature_cols]
X.drop(columns = ['is_fraud', 'year'], axis = 1, inplace = True)
y = transaction_cleaned['is_fraud']

# Split X & y into train and test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 97)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
## Use of SMOTE to upsample the data
ros = RandomOverSampler(random_state = 97)
X_train, y_train = ros.fit_resample(X_train, y_train)
print(f'Percentage of Fraudulent Transaction : {y_train.value_counts(normalize = True)[0] * 100}%')
print(f'Percentage of Normal Transaction : {y_train.value_counts(normalize = True)[1] * 100}%')

In [ ]:
## Use of MinMaxScaler to Scale data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("\nClassification Report: \n",classification_report(y_test, y_pred))
    print(f"Recall: {recall_score(y_test, y_pred)}")
    print("AUC:", roc_auc_score(y_test, y_pred))
    print(f"F1-Score: {f1_score(y_test, y_pred)}")
    print('')
          
    
    # ROC Curve
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.show()

    cm = confusion_matrix(y_test, y_pred, labels = model.classes_)
    disp = ConfusionMatrixDisplay(cm,display_labels= model.classes_)
    disp.plot()
    plt.show()

In [ ]:
# Initialize and fit Random Forest classifier for anomaly detection
rf = RandomForestClassifier(n_estimators=100, random_state=42)

print(f'Evaluation on Random Forest Classifier\n')
evaluate_model(rf, X_train, X_test, y_train, y_test)

In [ ]:
# Initialize and fit AdaBoost classifier for anomaly detection
ada = AdaBoostClassifier(n_estimators=100, random_state=42)

print(f'Evaluation on AdaBoost Classifier\n')
evaluate_model(ada, X_train, X_test, y_train, y_test)

In [ ]:
# Initialize and fit Gradient Boosting classifier for anomaly detection
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

print(f'Evaluation on Gradient Boosting\n')
evaluate_model(gb, X_train, X_test, y_train, y_test)

In [ ]:
# Initialize and fit Decision Tree classifier for anomaly detection
dt = DecisionTreeClassifier(random_state=42)

print(f'Evaluation on Decision Tree\n')
evaluate_model(dt, X_train, X_test, y_train, y_test)

Knowing that we have dealt with our problem with supervised learning, I will now attempt to use neural network to draw some extensive conclusion.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential() 
model.add(Dense(128, activation='relu', input_dim=20))
model.add(Dense(1, activation='sigmoid')) 
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy']) 
model.summary()

In [ ]:
# Fit the Neural Network
hist = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=100)

In [ ]:
sns.set()
acc = hist.history['accuracy']
val = hist.history['val_accuracy']
epochs = range(1, len(acc) + 1)

plt.plot(epochs, acc, '-', label='Training accuracy')
plt.plot(epochs, val, ':', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.plot()

A typical accuracy score computed by divding the sum of the true positives and true negatives by the number of test samples isn't very helpful because the dataset is so imbalanced. Fraudulent transactions represent less than 0.2% of all the samples, which means that the model could simply guess that every transaction is legitimate and get it right about 99.8% of the time. However knowing that I have upsampled the label data so that the distribution is 50 50, the model accuracy shows some impressive result.